In [ ]:
# !pip install numba
# !pip install pymap3d

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from eo_tools.util import show_cog
from eo_tools.S1_dev import fetch_dem_burst, geocode_burst, resample_burst_ampl
import numpy as np
from scipy.interpolate import griddata
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
data_dir = "/data/S1"
master_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20230903T183344_20230903T183412_050167_0609B4_100E.SAFE"
slave_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20230915T183345_20230915T183413_050342_060F9F_85A4.SAFE"
master_re = "/data/res/master_re.tiff"
slave_re = "/data/res/slave_re.tiff"
burst_idx = 5
iw = 2
pol = "vv"

In [ ]:
file_dem = fetch_dem_burst(master_dir, iw=iw, pol=pol, burst_idx=burst_idx)

In [ ]:
az_mst, rg_mst, dem_prof = geocode_burst(
    master_dir, file_dem, iw=iw, pol=pol, burst_idx=burst_idx
)

In [ ]:
az_slv, rg_slv, dem_prof = geocode_burst(
    slave_dir, file_dem, iw=iw, pol=pol, burst_idx=burst_idx
)

In [ ]:
resample_burst_ampl(
    master_dir,
    master_re,
    az_mst,
    rg_mst,
    dem_prof,
    iw=iw,
    pol=pol,
    burst_idx=burst_idx,
    order=3,
)

In [ ]:
resample_burst_ampl(
    slave_dir,
    slave_re,
    az_slv,
    rg_slv,
    dem_prof,
    iw=iw,
    pol=pol,
    burst_idx=burst_idx,
    order=3,
)

In [ ]:
from folium import Map, LayerControl
m = Map()
show_cog(master_re, m, rescale="3,8.1", expression="log(b1)")
show_cog(slave_re, m, rescale="3,8.1", expression="log(b1)")
LayerControl().add_to(m)
m

In [ ]:
# interpolate azimuth and range to fixed grid
from eo_tools.S1_dev import read_metadata

dir_tiff = Path(f"{master_dir}/measurement/")
dir_xml = Path(f"{master_dir}/annotation/")
pth_xml = dir_xml.glob(f"*iw{iw}*{pol}*.xml")
pth_tiff = dir_tiff.glob(f"*iw{iw}*{pol}*.tiff")
pth_xml = list(pth_xml)[0]
pth_tiff = list(pth_tiff)[0]

meta_mst = read_metadata(pth_xml)

In [ ]:
naz = int(meta_mst["product"]["swathTiming"]["linesPerBurst"])
nrg = int(meta_mst["product"]["swathTiming"]["samplesPerBurst"])

pixel spacing for IW:  
rg x az : 2.3 x 14.1 m
resize to get spacing comparable to DEM? -> much faster gridding

In [ ]:
grg, gaz = np.meshgrid(np.arange(0,nrg,16), np.arange(0,naz,8))

In [ ]:
plt.imshow(grg)

In [ ]:
cnd1 = (rg_mst >= 0) & (rg_mst < nrg)
cnd2 = (az_mst >= 0) & (az_mst < naz)
valid = cnd1 & cnd2 & (az_slv!=np.nan) & (rg_slv!=np.nan)

In [ ]:
coords = np.stack((az_mst[valid], rg_mst[valid])).T

In [ ]:
az_re = griddata(coords, az_slv[valid], (gaz.flat, grg.flat))

In [ ]:
plt.imshow(az_re.reshape(*gaz.shape))

In [ ]:
toto = np.zeros((naz, nrg))

In [ ]:
toto[az_mst[valid].astype(int), rg_mst[valid].astype(int)] = 1

In [ ]:
plt.figure(figsize=(20,80))
plt.imshow(toto.T, interpolation='bilinear')
plt.axis("off")

In [ ]:
np.median(az_mst[valid])

In [ ]:
np.arange(0,naz,16)

In [ ]:
gaz.shape